# GTSRB — chạy trên Google Colab

**Cách dùng: bật GPU rồi bấm `Runtime → Run all`. Hết.**

Notebook tự làm mọi thứ: tải code, tải dữ liệu, chạy 30 run ablation, chạy thêm seed,
đánh giá, rồi tải kết quả về máy bạn.

| | |
|---|---|
| Thời gian | **~2–2,5 giờ** trên T4 (trên MacBook là ~6 giờ) |
| Bạn phải bấm | **2 lần**: bật GPU, và cho phép truy cập Drive |
| Mất session giữa chừng | không sao — chạy lại Run all, nó bỏ qua phần đã xong |

---

## ⚠️ Làm ĐÚNG 2 việc này trước khi Run all

**1. Bật GPU**
`Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU` → `Save`

**2. Bấm Run all**
`Runtime` → `Run all`  (hoặc `Ctrl+F9`)

Khi chạy tới ô Drive, sẽ hiện popup xin quyền → bấm **Connect to Google Drive** →
chọn tài khoản → kéo xuống → **Allow**. Đó là lần bấm duy nhất.

> Nếu quên bật GPU, ô đầu tiên sẽ dừng lại và báo rõ — không chạy nhầm hàng chục tiếng trên CPU.

## 1 — Kiểm GPU

Ô này **cố ý dừng cả notebook** nếu chưa bật GPU, để bạn không chạy nhầm trên CPU.

In [ ]:
import torch, subprocess

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "\n>>> CHƯA BẬT GPU — notebook dừng ở đây. <<<\n\n"
        "Làm: Runtime -> Change runtime type -> T4 GPU -> Save\n"
        "Rồi bấm lại Runtime -> Run all.\n\n"
        "Chạy ablation trên CPU mất hàng chục tiếng thay vì 2 tiếng."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu}  ({vram:.0f} GB VRAM)")
print("\nOK, chạy tiếp.")

## 2 — Google Drive  ← *đây là chỗ bạn cần bấm*

Kết quả lưu lên Drive nên mất session không mất gì. Chạy lại Run all là tiếp tục
từ chỗ dở.

Popup hiện ra → **Connect to Google Drive** → chọn tài khoản → kéo xuống → **Allow**.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/gtsrb")
(DRIVE / "artifacts" / "runs").mkdir(parents=True, exist_ok=True)

done = [d for d in sorted((DRIVE / "artifacts" / "runs").glob("*"))
        if (d / "result.json").exists()]
print(f"Thư mục Drive : {DRIVE}")
print(f"Run đã có     : {len(done)}")
if done:
    print("  (những run này sẽ được BỎ QUA, không chạy lại)")

## 3 — Tải code

Lấy thẳng từ GitHub nên không phải upload file gì. Chạy lại thì nó `git pull` để
lấy bản mới nhất.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO = "https://github.com/tuanphongnguyen084/gtsrb-model-comparison.git"
PROJECT = Path("/content/DLprj")

if (PROJECT / ".git").exists():
    print("Đã có code, cập nhật bản mới nhất...")
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--quiet"], check=False)
else:
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    print("Tải code từ GitHub...")
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO, str(PROJECT)],
                   check=True)

os.chdir(PROJECT)

# artifacts/ trỏ sang Drive -> kết quả sống sót khi mất session
local = PROJECT / "artifacts"
if not local.is_symlink():
    if local.exists():
        shutil.rmtree(local)
    local.symlink_to(DRIVE / "artifacts")

# Kiểm symlink thật sự ghi được sang Drive
probe = local / ".probe"
probe.write_text("ok")
assert (DRIVE / "artifacts" / ".probe").read_text() == "ok", "symlink hỏng"
probe.unlink()

print("thư mục làm việc :", os.getcwd())
print("artifacts/       -> Drive  (đã kiểm ghi được)")

## 4 — Cài gói còn thiếu

Chỉ `thop` và `onnx`. Mọi thứ khác Colab có sẵn.

★ **Không `pip install torch`** — sẽ kéo về bản không khớp CUDA và bạn train trên CPU
mà không hề biết.

In [ ]:
!pip install -q -r requirements-colab.txt
import thop, torch
print("thop", thop.__version__, "| torch", torch.__version__,
      "| CUDA vẫn OK:", torch.cuda.is_available())

## 5 — Tải và tiền xử lý dữ liệu  (~5 phút)

Tải thẳng trên Colab (mạng ~100 MB/s) chứ không upload từ máy bạn.

Dữ liệu để ở `/content` chứ **không** để trên Drive: đọc từ Drive lúc train chậm hơn
nhiều vì Drive gắn qua mạng. Chỉ kết quả mới đáng để trên Drive.

In [ ]:
!python scripts/prepare_data.py --download --preprocess clahe --img-size 48

Ô dưới **chặn** notebook nếu dữ liệu không sạch — thà dừng còn hơn chạy 2 tiếng
rồi phát hiện số liệu vô giá trị.

In [ ]:
import sys; sys.path.insert(0, "src")

import subprocess, sys

def chay(*lenh):
    """Chạy một lệnh và DỪNG notebook nếu nó lỗi.

    ★ VÌ SAO KHÔNG DÙNG `!python ...` ★
    Với `!python`, lệnh lỗi chỉ in thông báo rồi notebook CHẠY TIẾP. Đúng điều
    đã xảy ra lần trước: ô ablation lỗi ngay (tham số --set chưa tồn tại trong
    run_ablation.py), Run all đi tiếp sang ô train M3, và sau 2 giờ ta có 3 run
    M3 mà KHÔNG có run ablation nào — mà không có gì báo động.
    """
    print("$", " ".join(lenh), flush=True)
    ma = subprocess.run(list(lenh)).returncode
    if ma != 0:
        raise SystemExit(
            f"\n{'='*64}\n  LỆNH LỖI (mã thoát {ma}) — DỪNG tại đây.\n"
            f"  {' '.join(lenh)}\n"
            f"  Sửa lỗi trên rồi Run all lại. Việc đã xong nằm trên Drive\n"
            f"  nên sẽ được bỏ qua, không chạy lại.\n{'='*64}")
    print("   -> OK", flush=True)


## 6 — ABLATION 30 run  (~1,5–2 giờ)

Đây là phần lâu nhất. Sáu trục: độ phân giải, augmentation, tiền xử lý, model scaling,
label smoothing, và từng thành phần kiến trúc của M2.

Dùng **ngân sách 15 epoch** — đủ để XẾP HẠNG các biến thể, tiết kiệm ~60% thời gian.

★ Chạy lại an toàn: run nào đã có `result.json` trên Drive sẽ bị bỏ qua.

In [ ]:
chay("python", "scripts/run_ablation.py", "--axes", "all", "--budget",
     "--set", "train.num_workers=2")


## 7 — Thêm 2 seed cho M2  (~25 phút)

Để trả lời: *chênh lệch giữa các model có vượt nhiễu khởi tạo không?*
Hiện nhóm mới chạy một seed — đó là hạn chế phương pháp lớn nhất.

In [ ]:
chay("python", "scripts/run_seeds.py", "--config", "configs/m2_vggres.yaml",
     "--seeds", "43", "44", "--set", "train.num_workers=2")


## 8 — Train M3 (TẮT mặc định)

Phiên Colab 4/10 đã train lại 3 model M3 **hai lần**, mỗi lần ~100 phút, cho ra
kết quả **giống y nguyên** (val macro-F1 0,94698 / 0,94629 / 0,94996 — khớp đến 5
chữ số). Và đo trên tập test chính thức thì chúng **kém hơn** run local:

| model | Colab | local |
|---|---|---|
| m3_resnet18 | 0,98801 | **0,99030** |
| m3_effnetb0 | 0,98255 | **0,98375** |
| m3_mobilenetv2 | 0,97647 | **0,97962** |

Nên ô này **tắt**. Thời gian Colab có hạn — dành hết cho ablation và seed, là
hai thứ KHÔNG chạy được ở local. Đặt `TRAIN_M3 = True` nếu thật sự cần.


In [ ]:
TRAIN_M3 = False     # đổi thành True nếu thật sự cần train lại M3 trên Colab

from pathlib import Path

NEEDED = ["m3_resnet18", "m3_mobilenetv2", "m3_effnetb0"]
co = {n: sorted(d.name for d in Path("artifacts/runs").glob(f"{n}_s42_2*")
                if (d / "result.json").exists()) for n in NEEDED}
for n, ds in co.items():
    print(f"  {n:16} {len(ds)} run" + (f"  ({ds[-1]})" if ds else ""))

if not TRAIN_M3:
    print("\nTRAIN_M3 = False -> bỏ qua. Dùng run M3 ở máy local (tốt hơn).")
else:
    thieu = [n for n, ds in co.items() if not ds]
    if not thieu:
        print("\nĐủ cả 3 backbone M3 rồi — bỏ qua.")
    for n in thieu:
        print(f"\n>>> train {n}")
        chay("python", "scripts/train.py", "--config", f"configs/{n}.yaml",
             "--set", "train.num_workers=2", "train.batch_size=96")


## 9 — Đánh giá trên tập test

Bắt buộc — không có số test thì bảng ablation trống. Bỏ McNemar vì phần đó chạy
trên máy cá nhân khi tổng hợp.

In [ ]:
!python scripts/evaluate.py --runs "artifacts/runs/*" --no-mcnemar

## 10 — Tải kết quả về máy

Chỉ đóng gói `result.json`, `config.yaml`, `train_log.csv` — **không lấy checkpoint**.
Ablation chỉ cần số liệu, mà 30 checkpoint là hàng GB. Gói này chỉ vài trăm KB.

File sẽ tự tải xuống thư mục `Downloads` của bạn.

In [ ]:
import zipfile
from pathlib import Path

KEEP = ("result.json", "config.yaml", "train_log.csv")
out = Path("/content/ketqua_colab.zip")

runs = [r for r in sorted(Path("artifacts/runs").glob("*"))
        if (r / "result.json").exists()]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for run in runs:
        for name in KEEP:
            p = run / name
            if p.exists():
                z.write(p, f"artifacts/runs/{run.name}/{name}")

print(f"Đóng gói {len(runs)} run -> {out.name}  ({out.stat().st_size/1024:.0f} KB)")

# Thống kê nhanh cho biết đã có đủ chưa
import re
axes = ("resolution", "augmentation", "preprocess", "scaling",
        "label_smoothing", "components")
n_abl = sum(1 for r in runs if any(a in r.name for a in axes))
n_seed = sum(1 for r in runs if "seed4" in r.name)
print(f"  ablation : {n_abl}/30")
print(f"  seed     : {n_seed}")
print(f"  model chính: {sum(1 for r in runs if not any(a in r.name for a in axes) and 'seed4' not in r.name)}")

from google.colab import files
files.download(str(out))
# ---- Kiểm trước khi đóng gói: thiếu thì nói THẲNG ----
#
# Lần trước ô này in "ablation : 0/30" mà lẫn giữa các dòng khác nên trôi qua,
# và cả mẻ 2 giờ coi như mất. Giờ thiếu là in banner không thể bỏ sót.
thieu = []
if n_abl < 30:
    thieu.append(f"ablation {n_abl}/30")
if n_seed < 2:
    thieu.append(f"seed {n_seed}/2")
if thieu:
    print("\n" + "!" * 66)
    print("  CHƯA ĐỦ: " + " · ".join(thieu))
    print("  Phiên Colab hay bị ngắt. BẤM RUN ALL LẠI — việc đã xong nằm trên")
    print("  Drive nên sẽ được bỏ qua, chỉ chạy phần còn thiếu.")
    print("  Nếu một ô báo lỗi và notebook DỪNG: copy thông báo lỗi đó gửi lại.")
    print("!" * 66)
else:
    print("\n*** ĐỦ ablation 30/30 và seed 2/2 — tải zip về máy. ***")


---

# Xong — làm gì tiếp trên máy bạn

```bash
cd ~/Downloads/DLprj
python scripts/merge_runs.py ~/Downloads/ketqua_colab.zip

python scripts/run_ablation.py --collect-only   # bảng + 6 hình ablation
python scripts/run_seeds.py --collect-only      # mean±std + kết luận nhiễu seed
make report baocao                              # sinh lại 2 báo cáo
```

`merge_runs.py` chỉ chép run chưa có và kiểm `result.json` hợp lệ trước khi chép.

---

## Nếu có trục trặc

| Hiện tượng | Xử lý |
|---|---|
| Ô 1 dừng, báo chưa có GPU | `Runtime → Change runtime type → T4 GPU → Save`, rồi Run all lại |
| Mất kết nối giữa chừng | Bấm Run all lại. Kết quả nằm trên Drive, nó bỏ qua phần đã xong |
| Colab báo hết hạn GPU | Bản miễn phí giới hạn ~12h/ngày. Hôm sau Run all lại, kết quả vẫn còn |
| Tab bị đóng | Colab chạy thêm ~90 phút rồi ngắt. Mở lại tab để giữ phiên |

**Trong lúc chạy: đừng đóng tab, đừng để máy ngủ.**

## Vì sao không đo tốc độ ở đây

Latency của T4 không nói gì về thiết bị biên. `make speed` và `make edge` phải chạy
trên **máy đích**, và lúc đo phải tắt mọi thứ khác — nên hai phần đó giữ ở máy cá nhân.